ASSIGNMENT NLP – 4 (BERT Fine-Tuning) : Fine-Tuning BERT on a Kaggle Dataset

In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Data Preprocessing (Mandatory)
print("Loading and preprocessing data...")
df = pd.read_csv("IMDB Dataset.csv")

# Clean and preprocess text data (basic HTML tag removal and lowercase)
df['review'] = df['review'].str.replace('<br />', ' ')
df['review'] = df['review'].str.lower()

# Handle missing values if any
df = df.dropna()

# Convert sentiment to binary labels
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df = df.drop(columns=['sentiment'])

# Note: Taking a smaller sample (e.g., 5000) is highly recommended for local VS Code testing
df = df.sample(5000, random_state=42) 

# 2. Data Splitting
# Split dataset into Train, Validation, and Test sets
train_texts, temp_texts, train_labels, temp_labels = train_test_split(df['review'].tolist(), df['label'].tolist(), test_size=0.3, random_state=42)
val_texts, test_texts, val_labels, test_labels = train_test_split(temp_texts, temp_labels, test_size=0.5, random_state=42)

print(f"Train size: {len(train_texts)}, Validation size: {len(val_texts)}, Test size: {len(test_texts)}")

Matplotlib is building the font cache; this may take a moment.
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and preprocessing data...
Train size: 3500, Validation size: 750, Test size: 750


In [2]:
# 3. Tokenization
# Use bert-base-uncased tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256)

# Create a PyTorch Dataset class
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)

C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.show()

In [4]:
print("--- Experiment 1: Freezing BERT layers and training classifier ---")

# 4. Model Building
model_frozen = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Freeze BERT layers and train classifier
for param in model_frozen.bert.parameters():
    param.requires_grad = False

# 5. Fine-Tuning Setup
# The Hugging Face Trainer uses the AdamW optimizer by default
training_args = TrainingArguments(
    output_dir='./results_frozen',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5, # Explicitly setting required learning rate
    evaluation_strategy="epoch",
    logging_dir='./logs',
)

trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train on dataset
trainer_frozen.train()

# 6. Model Evaluation
results_frozen = trainer_frozen.evaluate(test_dataset)
print("Experiment 1 Metrics:", results_frozen)

# Generate Predictions for Confusion Matrix
preds_frozen = trainer_frozen.predict(test_dataset)
plot_confusion_matrix(test_labels, preds_frozen.predictions.argmax(-1), "Confusion Matrix - Frozen BERT")

--- Experiment 1: Freezing BERT layers and training classifier ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2528.69it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
print("--- Experiment 2: Fine-tuning last 2 layers of BERT ---")

model_unfrozen = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Freeze all layers first
for param in model_unfrozen.bert.parameters():
    param.requires_grad = False

# Fine-tune last 2 layers of BERT
for param in model_unfrozen.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

training_args_unfrozen = TrainingArguments(
    output_dir='./results_unfrozen',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    logging_dir='./logs_unfrozen',
)

trainer_unfrozen = Trainer(
    model=model_unfrozen,
    args=training_args_unfrozen,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train and Evaluate
trainer_unfrozen.train()
results_unfrozen = trainer_unfrozen.evaluate(test_dataset)
print("Experiment 2 Metrics:", results_unfrozen)

# Confusion Matrix for Unfrozen Layers
preds_unfrozen = trainer_unfrozen.predict(test_dataset)
plot_confusion_matrix(test_labels, preds_unfrozen.predictions.argmax(-1), "Confusion Matrix - Unfrozen Last 2 Layers")

In [ ]:
# Detailed analysis and comparison
print("\n--- Final Performance Comparison ---")
print(f"Frozen Layers - Accuracy: {results_frozen['eval_accuracy']:.4f}, F1: {results_frozen['eval_f1']:.4f}")
print(f"Unfrozen Last 2 Layers - Accuracy: {results_unfrozen['eval_accuracy']:.4f}, F1: {results_unfrozen['eval_f1']:.4f}")

"""
Analysis & Insights:
By freezing all layers (Experiment 1), the model trains quickly but relies solely on the raw pre-trained embeddings, which may yield slightly lower accuracy. 
By fine-tuning the last 2 layers (Experiment 2), the model adapts its highest-level contextual representations to the specific sentiment analysis task. This generally results in superior performance across Accuracy, Precision, Recall, and F1 Score, demonstrating the value of task-specific adaptation in transformer models.
"""